# Judge Prompt Test Bench

A sandbox for iterating on the LLM-as-judge prompt. Two evaluation modes, run side by side on the **same trajectories**:

1. **Grounded (absolute)** — the judge reads one resident's trajectory and scores it 1–5.
2. **Pairwise (relative)** — the judge compares two residents' trajectories (same person, same events, different system config) and picks the better one.

**Key design choice:** every dimension (Behavioral Plausibility, Persona Consistency, Intervention Responsiveness) is scored in a **separate API call**. This decorrelates the dimensions (no halo effect from scoring them together) and lets you tune one dimension's rubric at a time.

## How to use it
1. Run **Setup** and **Load data** once.
2. Edit the prompt strings in **§3 Prompts** — the shared preamble and the three per-dimension rubrics. These are the only cells you normally change.
3. Run **§4 Grounded** and **§5 Pairwise**. Each prints a budget estimate and asks nothing else.
4. Read the **diagnostics** after each: does the scale open up? do conditions separate? how noisy / order-stable is it?
5. Change a prompt, re-run, compare. The grounded diagnostics also show a **delta vs the committed scores** in `outputs/eval/fullsim_scores_claude.csv`.

Nothing here writes to the committed caches or the production `client.py`. It is a test bench only.

## 1. Setup

In [ ]:
# Imports and API keys. Run once. (Model + clients are chosen in the next cell.)
import json, os, sys, re, itertools
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# repo root (notebook lives in notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# load API keys from env.local (never committed) into the environment
env_file = ROOT / "env.local"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

from src.llm.client import (
    Config, init_openrouter_client, init_clients,
    _call_llm, _strip_fences, usage_tracker, MODEL_PRICING,
)

print("repo root:", ROOT)
print("OPENROUTER_API_KEY present:", "OPENROUTER_API_KEY" in os.environ)
print("ANTHROPIC_API_KEY  present:", "ANTHROPIC_API_KEY" in os.environ)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG — this is the cell you edit to choose the judge model and test scope.
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. PICK THE JUDGE MODEL ───────────────────────────────────────────────────
# The simulations in outputs/runs/ are the CLAUDE family. To avoid a model grading
# its own outputs (self-preference bias), judge them with a NON-Claude model.
#
# Available models (id : family, provider, price $/M in-out) — all priced in client.py:
#
#     openai/gpt-5.4         OpenAI, OpenRouter   $2.50 / $15.00   ← default, cross-family
#     openai/gpt-5.4-mini    OpenAI, OpenRouter   $0.75 /  $4.50   cheaper, weaker
#     openai/gpt-5.4-nano    OpenAI, OpenRouter   $0.20 /  $1.25   cheapest, weakest
#     moonshotai/kimi-k2.6   Moonshot, OpenRouter $0.66 /  $3.41   another cross-family option
#     claude-opus-4-6        Anthropic, native    $5.00 / $25.00   ⚠ SAME family as the sims
#     claude-sonnet-4-6      Anthropic, native    $3.00 / $15.00   ⚠ SAME family as the sims
#
# Just change the string below. Anything starting with "claude-" is routed to the
# Anthropic client automatically; everything else goes to OpenRouter.
JUDGE_MODEL = "openai/gpt-5.4"

# ── 2. JUDGE CALL SETTINGS ────────────────────────────────────────────────────
JUDGE_TEMP    = 0.0     # 0 = deterministic / reproducible. Raise only to study judge noise.
JUDGE_MAX_TOK = None    # None = auto (8192 for GPT-5.x reasoning models, else 1024). Or set an int.
PARALLELISM   = 8       # concurrent judge calls

# ── 3. TEST SCOPE (controls how many calls, i.e. cost) ────────────────────────
REP     = 1       # which replicate of each condition to test on (1 = cheapest)
AGENTS  = None    # None = all 5 residents; or a subset e.g. ["beth", "edward"]

# ── 4. FIXED PATHS / LABELS (rarely need changing) ────────────────────────────
SIM_FAMILY = "claude"   # which simulation family's runs to judge (matches outputs/runs/)
RUNS_DIR       = ROOT / "outputs/runs"
AGENT_DIR      = ROOT / "config/agents/selected"
CACHED_FULLSIM = ROOT / f"outputs/eval/fullsim_scores_{SIM_FAMILY}.csv"   # committed scores, for benchmarking

VARIANTS    = ["Baseline", "Ablation1_No_Reflection", "Ablation2_No_Memory_No_Reflection", "Budget"]
BASELINE    = "Baseline"
CHALLENGERS = ["Ablation1_No_Reflection", "Ablation2_No_Memory_No_Reflection", "Budget"]  # pairwise: each vs Baseline

DIMS = ["behavioral_plausibility", "persona_consistency", "intervention_responsiveness"]
DIM_LABEL = {"behavioral_plausibility": "Behavioral Plausibility",
             "persona_consistency": "Persona Consistency",
             "intervention_responsiveness": "Intervention Responsiveness"}

# ══════════════════════════════════════════════════════════════════════════════
#  Below here: validation, client init, and a summary. You don't need to edit it.
# ══════════════════════════════════════════════════════════════════════════════
def _family_of(model: str) -> str:
    if model.startswith("claude-"):     return "claude"
    if model.startswith("openai/"):     return "openai"
    return model.split("/")[0] if "/" in model else "other"

_is_claude_judge = JUDGE_MODEL.startswith("claude-")

# auto max_tokens: GPT-5.x spends reasoning tokens against the budget, so give it headroom
if JUDGE_MAX_TOK is None:
    JUDGE_MAX_TOK = 8192 if _family_of(JUDGE_MODEL) == "openai" else 1024

# init only the client the chosen model needs
orc = None
anthro = None
_warnings = []
if _is_claude_judge:
    anthro = init_clients()
else:
    orc = init_openrouter_client()

# validation / friendly warnings
if JUDGE_MODEL not in MODEL_PRICING:
    _warnings.append(f"'{JUDGE_MODEL}' is not in MODEL_PRICING — cost estimates will read $0. "
                     f"Add it to MODEL_PRICING in client.py, or pick a listed model.")
if _family_of(JUDGE_MODEL) == SIM_FAMILY:
    _warnings.append(f"Judge family ({_family_of(JUDGE_MODEL)}) matches the simulation family "
                     f"({SIM_FAMILY}). This is SELF-JUDGING and risks self-preference bias. "
                     f"Prefer a cross-family judge (e.g. openai/gpt-5.4).")
if JUDGE_TEMP != 0.0:
    _warnings.append(f"JUDGE_TEMP = {JUDGE_TEMP} (not 0) — scores will not be reproducible run to run.")

_in, _out = MODEL_PRICING.get(JUDGE_MODEL, (0.0, 0.0))
print("┌─ JUDGE CONFIG ───────────────────────────────────────────────")
print(f"│  model        : {JUDGE_MODEL}  ({_family_of(JUDGE_MODEL)}, "
      f"{'Anthropic' if _is_claude_judge else 'OpenRouter'})")
print(f"│  price $/M    : in ${_in}  out ${_out}")
print(f"│  temperature  : {JUDGE_TEMP}")
print(f"│  max_tokens   : {JUDGE_MAX_TOK}")
print(f"│  judging      : {SIM_FAMILY} sims  |  replicate {REP}  |  "
      f"agents = {'all' if AGENTS is None else AGENTS}")
print("└──────────────────────────────────────────────────────────────")
for w in _warnings:
    print("⚠ ", w)
if not _warnings:
    print("✓ config looks good")

## 2. Load data
Loads the chosen replicate of every condition and reshapes it into per-resident trajectories plus each resident's seed narrative and memory seeds (the same context the production judge receives).

In [ ]:
def load_agent_configs(agent_dir: Path) -> dict:
    cfgs = {}
    for p in sorted(agent_dir.glob("*.yaml")):
        raw = yaml.safe_load(p.read_text())
        d = raw["agents"][0] if isinstance(raw.get("agents"), list) else raw
        cfgs[d["id"]] = d
    return cfgs

def format_seeds(memory_seeds) -> str:
    if not memory_seeds:
        return "(none)"
    return "\n".join(f"- {s['description']}" for s in memory_seeds if isinstance(s, dict) and s.get("description"))

def format_trajectory(decisions) -> str:
    return "\n\n".join(
        f"--- Day {d['day']}: {d['event_type'].upper()} ---\n"
        f"INTERVENTION: {d['intervention']}\n"
        f"RESIDENT DECISION: {d['decision']}\n"
        f"RESIDENT REASONING: {d['reasoning']}"
        for d in decisions
    )

agent_cfgs = load_agent_configs(AGENT_DIR)

# variant -> agent_id -> list of decision dicts (one resident's full trajectory)
traj = defaultdict(dict)
for f in sorted(RUNS_DIR.glob(f"*_rep{REP}_*.jsonl")):
    entries = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    rc = next(e for e in entries if e.get("entry_type") == "run_config")
    variant = rc["run_label"].replace("claude_", "").rsplit("_rep", 1)[0]
    if variant not in VARIANTS:
        continue
    by_agent = defaultdict(list)
    for e in entries:
        if e.get("entry_type") == "decision":
            by_agent[e["agent_id"]].append(e)
    for aid, evs in by_agent.items():
        if AGENTS and aid not in AGENTS:
            continue
        evs.sort(key=lambda e: e["tick"])
        traj[variant][aid] = [{"day": e["tick"], "event_type": e["event_type"],
                               "intervention": e["intervention"], "decision": e["decision"],
                               "reasoning": e["reasoning"], "display": e["agent_display_name"]} for e in evs]

agent_ids = sorted(traj[BASELINE].keys())
print(f"replicate {REP} | {len(traj)} conditions | {len(agent_ids)} agents: {agent_ids}")
for v in VARIANTS:
    n = sum(len(t) for t in [traj[v].get(a, []) for a in agent_ids])
    print(f"  {v:<36} {len(traj[v])} agents, {n} decisions")

## 3. Prompts — *edit these*

Everything the judge sees is built from the strings below. The dimension rubrics are **per-dimension**: each call contains the shared preamble plus exactly one rubric, so the judge scores one thing at a time.

- `PREAMBLE_*` — shared role/context, one for grounded and one for pairwise.
- `RUBRIC[dim]` — grounded 1–5 anchors for one dimension.
- `PAIRWISE_Q[dim]` — the comparison question for one dimension.

Tune, re-run §4 / §5, compare diagnostics.

In [ ]:
# ── SHARED PREAMBLES ──────────────────────────────────────────────────────────
PREAMBLE_GROUNDED = (
    "You are an expert evaluator assessing a resident's behaviour across a wildfire "
    "mitigation study. People include homeowners, renters, and people with varied "
    "relationships to the property. Read the seed narrative carefully to understand "
    "each resident's actual role and constraints before scoring.\n\n"
    "Plausibility means plausible for THIS resident given who they are, not plausible "
    "for a generic cooperative homeowner. Compliance, refusal, deflection and reframing "
    "are all legitimate: judge whether THIS resident would respond this way, never "
    "whether the response is cooperative.\n\n"
    "You are scoring ONE criterion only. Ignore the other criteria. "
    "Be critical and discriminating: use the full 1 to 5 range."
)

PREAMBLE_PAIRWISE = (
    "You are an expert evaluator comparing two accounts of how the SAME resident "
    "responded to the SAME sequence of wildfire mitigation events. The two accounts come "
    "from different sources; you know nothing else about them.\n\n"
    "Judge whether THIS resident, with their specific history and constraints, is more "
    "convincingly present. Compliance, refusal, deflection and reframing are all "
    "legitimate: never reward an account simply for being more cooperative.\n\n"
    "You are comparing on ONE criterion only. If you genuinely cannot tell the two apart "
    "on this criterion, answer \"tie\" — but if one is even slightly better, say so."
)

# ── GROUNDED RUBRICS (1 to 5), one per dimension ──────────────────────────────
RUBRIC = {
"behavioral_plausibility": (
    "BEHAVIORAL PLAUSIBILITY — How plausible is this resident's behaviour across the "
    "trajectory given who they are and what constraints they face?\n"
    "  1 = Not at all plausible for this resident's situation\n"
    "  3 = Reasonable but generic — could apply to many people in a similar situation\n"
    "  5 = Reflects the specific competing pressures, constraints and worldview unique "
    "to this resident"
),
"persona_consistency": (
    "PERSONA CONSISTENCY — Would this trajectory look meaningfully different if the "
    "resident had a different seed narrative? If not, score 3 or below. Distinctive "
    "voice, tone and recurring framings count as persona signals, not just facts.\n"
    "  1 = Contradicts the seed personality or key memory seeds\n"
    "  3 = Consistent with this persona type, but the resident's specific history, named "
    "experiences or distinctive voice rarely surface\n"
    "  5 = The resident's unique history and voice are clearly present — references named "
    "people, places, costs or opinions that could only come from this seed"
),
"intervention_responsiveness": (
    "INTERVENTION RESPONSIVENESS — How specifically did the resident engage with the "
    "content of the interventions they received?\n"
    "  1 = Ignored intervention specifics or gave fully generic responses\n"
    "  3 = Acknowledged events appropriately but at a generic level, without integrating "
    "specific details\n"
    "  5 = Consistently integrated specific details from each intervention and connected "
    "them to prior context"
),
}

# ── PAIRWISE QUESTIONS, one per dimension ─────────────────────────────────────
PAIRWISE_Q = {
"behavioral_plausibility": (
    "In which account is the resident's behaviour more plausible given their specific "
    "circumstances, role and constraints (as opposed to plausible for a generic homeowner)?"
),
"persona_consistency": (
    "In which account is this resident's particular history, named experiences, costs, "
    "opinions and characteristic voice more clearly present? Which account would look more "
    "different if the seed narrative were swapped for someone else's?"
),
"intervention_responsiveness": (
    "In which account does the resident engage more specifically with the actual content "
    "of each intervention, and connect events to what came before?"
),
}

print("prompts defined:", list(RUBRIC), "| preamble grounded/pairwise ready")

In [ ]:
# ── Prompt builders (single dimension per call) ───────────────────────────────
def build_grounded(dim, seed, seeds_text, trajectory_text):
    system = PREAMBLE_GROUNDED
    user = (
        f"RESIDENT SEED NARRATIVE:\n{seed}\n\n"
        f"RESIDENT MEMORY SEEDS:\n{seeds_text}\n\n"
        f"FULL TRAJECTORY:\n{trajectory_text}\n\n"
        f"Score this resident on ONE criterion:\n\n{RUBRIC[dim]}\n\n"
        "First write a one to two sentence note that quotes an exact phrase from the "
        "trajectory as evidence, then give the score.\n\n"
        "Respond in this exact JSON format:\n"
        '{\n  "note": "<1-2 sentences with a quoted phrase>",\n  "score": <number 1-5>\n}'
    )
    return system, user

def build_pairwise(dim, seed, seeds_text, traj_a, traj_b):
    system = PREAMBLE_PAIRWISE
    user = (
        f"RESIDENT SEED NARRATIVE:\n{seed}\n\n"
        f"RESIDENT MEMORY SEEDS:\n{seeds_text}\n\n"
        f"===== ACCOUNT A =====\n{traj_a}\n\n"
        f"===== ACCOUNT B =====\n{traj_b}\n\n"
        f"Compare the two accounts on ONE criterion:\n\n{PAIRWISE_Q[dim]}\n\n"
        "First write one sentence quoting a phrase from each account, then the verdict "
        '("A", "B", or "tie").\n\n'
        "Respond in this exact JSON format:\n"
        '{\n  "note": "<one sentence quoting both>",\n  "verdict": "<A|B|tie>"\n}'
    )
    return system, user

def call_judge(system, user):
    """One judge call via the production transport (handles GPT-5.x temp quirk + token tracking)."""
    raw = _call_llm(model=JUDGE_MODEL, system=system, user=user,
                    max_tokens=JUDGE_MAX_TOK, temperature=JUDGE_TEMP,
                    client_anthropic=anthro, client_openrouter=orc, call_type="judge")
    return _strip_fences(raw)

def parse_json(raw):
    try:
        return json.loads(raw)
    except Exception:
        return {"_raw": raw}

def run_parallel(jobs, worker):
    """jobs: list of anything; worker(job)->row. Returns list of rows, prints progress."""
    rows, errs = [], []
    with ThreadPoolExecutor(max_workers=PARALLELISM) as pool:
        futs = {pool.submit(worker, j): j for j in jobs}
        for i, fut in enumerate(as_completed(futs), 1):
            try:
                rows.append(fut.result())
            except Exception as e:
                errs.append((futs[fut], repr(e)))
                print("  FAILED:", repr(e)[:120])
            if i % 20 == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)} done")
    if errs:
        print(f"{len(errs)} failures")
    return rows

def budget_note(n_calls):
    ir, orr = in_out_rate(JUDGE_MODEL)
    # rough: ~1.5k in + ~0.4k out per call is typical for these trajectories; adjust after first run
    est = n_calls * (1500*ir + 400*orr) / 1_000_000
    print(f"≈ {n_calls} judge calls  |  rough cost estimate ${est:.2f} "
          f"(refine from the printed actuals after running)")
print("builders ready")

## 4. Grounded evaluation
One call per (resident, dimension). With 4 conditions × 5 agents × 3 dimensions that is 60 calls for the standard scope.

In [ ]:
usage_tracker.reset()

grounded_jobs = []
for v in VARIANTS:
    for aid in agent_ids:
        decs = traj[v].get(aid)
        if not decs:
            continue
        seed = agent_cfgs.get(aid, {}).get("seed_narrative", "")
        seeds_text = format_seeds(agent_cfgs.get(aid, {}).get("memory_seeds", []))
        ttext = format_trajectory(decs)
        for dim in DIMS:
            grounded_jobs.append((v, aid, dim, seed, seeds_text, ttext))

budget_note(len(grounded_jobs))

def grounded_worker(job):
    v, aid, dim, seed, seeds_text, ttext = job
    sysmsg, usr = build_grounded(dim, seed, seeds_text, ttext)
    d = parse_json(call_judge(sysmsg, usr))
    score = d.get("score")
    try: score = float(score)
    except (TypeError, ValueError): score = np.nan
    return {"variant": v, "agent_id": aid, "dimension": dim, "score": score,
            "note": d.get("note"), "raw": d.get("_raw")}

g_rows = run_parallel(grounded_jobs, grounded_worker)
grounded = pd.DataFrame(g_rows)
print("\nactual judge cost:", usage_tracker.to_dict(agent_model=JUDGE_MODEL, judge_model=JUDGE_MODEL)["judge_cost_usd"], "USD")
grounded.head()

### 4a. Grounded diagnostics
Does the prompt open up the scale, and do the conditions separate? Compared against the committed scores where possible.

In [ ]:
# wide: one row per (variant, agent), columns = dimensions + overall
gw = grounded.pivot_table(index=["variant","agent_id"], columns="dimension", values="score").reset_index()
gw["overall"] = gw[DIMS].mean(axis=1)

print("="*70); print("SCALE USE (new prompt)"); print("="*70)
vals = sorted(pd.unique(grounded["score"].dropna()))
print("distinct scores used:", vals)
print(f"min {grounded['score'].min():.1f} | max {grounded['score'].max():.1f} | "
      f"mean {grounded['score'].mean():.2f} | share >=4.5: {(grounded['score']>=4.5).mean():.0%} | "
      f"NaN: {grounded['score'].isna().sum()}")
for dim in DIMS:
    vc = grounded[grounded.dimension==dim]["score"].value_counts().sort_index()
    print(f"  {DIM_LABEL[dim]:<30} " + "  ".join(f"{k}:{v}" for k,v in vc.items()))

print("\n"+"="*70); print("DISCRIMINATION (condition means)"); print("="*70)
cond = gw.groupby("variant")[DIMS+["overall"]].mean().reindex(VARIANTS)
print(cond.round(3).to_string())
spread = cond["overall"].max()-cond["overall"].min()
agent_spread = gw.groupby("agent_id")["overall"].mean().pipe(lambda s: s.max()-s.min())
print(f"\ncondition spread (overall): {spread:.3f}   agent spread (confound): {agent_spread:.3f}")

# benchmark vs committed scores
if CACHED_FULLSIM.exists():
    print("\n"+"="*70); print("BENCHMARK vs committed fullsim_scores_claude.csv (joint judge)"); print("="*70)
    old = pd.read_csv(CACHED_FULLSIM)
    old = old[old["rep"]==REP]
    old_vals = sorted(pd.unique(old[DIMS].values.ravel()))
    print(f"OLD  distinct {old_vals}  mean {old[DIMS].values.mean():.2f}  share>=4.5 {(old[DIMS].values>=4.5).mean():.0%}")
    print(f"NEW  distinct {vals}  mean {grounded['score'].mean():.2f}  share>=4.5 {(grounded['score']>=4.5).mean():.0%}")
    oc = old.assign(overall=old[DIMS].mean(axis=1)).groupby("variant")["overall"].mean().reindex(VARIANTS)
    cmp = pd.DataFrame({"OLD overall": oc, "NEW overall": cond["overall"]}).round(3)
    cmp["Δ"] = (cmp["NEW overall"]-cmp["OLD overall"]).round(3)
    print(cmp.to_string())
else:
    print("\n(no cached scores found to benchmark against)")

## 5. Pairwise evaluation
One call per (contrast, resident, ordering, dimension). Each Baseline-vs-challenger pair is judged in **both** orderings so position bias can be measured and cancelled. 3 contrasts × 5 agents × 2 orderings × 3 dims = 90 calls for the standard scope.

In [ ]:
usage_tracker.reset()

pair_jobs = []
for ch in CHALLENGERS:
    for aid in agent_ids:
        if aid not in traj[BASELINE] or aid not in traj[ch]:
            continue
        seed = agent_cfgs.get(aid, {}).get("seed_narrative", "")
        seeds_text = format_seeds(agent_cfgs.get(aid, {}).get("memory_seeds", []))
        base_t = format_trajectory(traj[BASELINE][aid])
        chal_t = format_trajectory(traj[ch][aid])
        for order in ("base_first", "chal_first"):
            for dim in DIMS:
                pair_jobs.append((ch, aid, order, dim, seed, seeds_text, base_t, chal_t))

budget_note(len(pair_jobs))

def pair_worker(job):
    ch, aid, order, dim, seed, seeds_text, base_t, chal_t = job
    a, b = (base_t, chal_t) if order == "base_first" else (chal_t, base_t)
    sysmsg, usr = build_pairwise(dim, seed, seeds_text, a, b)
    d = parse_json(call_judge(sysmsg, usr))
    v = str(d.get("verdict", "")).strip().lower()
    if v == "tie":
        winner = "tie"
    elif v in ("a", "b"):
        base_is_a = (order == "base_first")
        winner = (BASELINE if base_is_a else ch) if v == "a" else (ch if base_is_a else BASELINE)
    else:
        winner = None
    return {"contrast": ch, "agent_id": aid, "order": order, "dimension": dim,
            "verdict_pos": v, "winner": winner, "note": d.get("note")}

p_rows = run_parallel(pair_jobs, pair_worker)
pairwise = pd.DataFrame(p_rows)
print("\nactual judge cost:", usage_tracker.to_dict(agent_model=JUDGE_MODEL, judge_model=JUDGE_MODEL)["judge_cost_usd"], "USD")
pairwise.head()

### 5a. Pairwise diagnostics
Raw win rates, the trustworthy order-robust view (both orderings must agree), plus the two reliability checks: position bias and order-consistency.

In [ ]:
print("="*70); print("RAW WIN RATE — Baseline wins (both orderings pooled; 50% = no effect)"); print("="*70)
print(f"{'contrast':<36}" + "".join(f"{DIM_LABEL[d][:14]:<15}" for d in DIMS))
for ch in CHALLENGERS:
    sub = pairwise[pairwise.contrast==ch]; cells=[]
    for d in DIMS:
        dec = sub[(sub.dimension==d) & sub.winner.isin([BASELINE, ch])]
        w = (dec.winner==BASELINE).sum()
        cells.append(f"{w}/{len(dec)} ({w/len(dec)*100:.0f}%)" if len(dec) else "n/a")
    print(f"{ch:<36}" + "".join(f"{c:<15}" for c in cells))

print("\n"+"="*70); print("ORDER-ROBUST — both A/B orderings must agree (else unresolved). n = agents"); print("="*70)
print(f"{'contrast':<36}{'dim':<6}{'Base':<6}{'Chal':<6}{'unres'}")
for ch in CHALLENGERS:
    for d in DIMS:
        b=c=u=0
        for aid, gg in pairwise[(pairwise.contrast==ch)&(pairwise.dimension==d)].groupby("agent_id"):
            s = set(gg.winner)
            if len(s)==1:
                w=s.pop(); b+=w==BASELINE; c+=w==ch; u+=w not in (BASELINE,ch)
            else:
                u+=1
        print(f"{ch:<36}{d[:2].upper():<6}{b:<6}{c:<6}{u}")
    print()

print("="*70); print("RELIABILITY"); print("="*70)
ties = (pairwise.winner=="tie").sum()
print(f"ties: {ties}/{len(pairwise)} ({ties/len(pairwise)*100:.0f}%)")
posA = (pairwise.verdict_pos=="a").sum(); posB = (pairwise.verdict_pos=="b").sum()
print(f"position bias: A {posA} / B {posB}  ({posA/(posA+posB)*100:.0f}% A, want ~50%)")
agree=tot=0
for (ch,aid,d), gg in pairwise.groupby(["contrast","agent_id","dimension"]):
    if len(gg)!=2: continue
    tot+=1; agree += int(gg.iloc[0].winner == gg.iloc[1].winner)
print(f"order consistency: {agree}/{tot} ({(agree/tot*100) if tot else 0:.0f}%) verdicts survive A/B swap")

## 6. Reading the results

**Grounded — is the prompt healthy?**
- *Scale use:* more distinct values and a lower `share >=4.5` means less ceiling saturation. The committed joint judge used only 4.0 / 4.5 / 5.0 and never went below 4.0.
- *Discrimination:* a larger condition spread relative to the agent spread means the prompt is picking up on the manipulation rather than on which resident it is.

**Pairwise — is the comparison trustworthy?**
- *Order-robust* is the table to believe; raw win rates are inflated by position bias.
- *Position bias* near 50% and *order consistency* high (>80%) mean the verdicts are stable. The earlier pilot was 63% / 64%, which is why it stayed a pilot.

**Cost:** the actual judge cost prints after each run. Keep `REP = 1` and the standard agent set while iterating; only scale up once a prompt looks good.